# Stage 3 — Narrative Generation QC

Compare rule-based and Cortex LLM narratives side-by-side.

| Step | What | What to check |
|------|------|---------------|
| 0 | Config | Dates, sample size |
| 1 | Session + data + detect + enrich | Enriched signals |
| 2 | Rule-based narratives | All four fields populated, no blanks |
| 3 | Full narrative print | Read through top signal |
| 4 | Cortex narrative (single signal) | Compare with rule-based |
| 5 | Cortex batch (first N) | Success/fail count |

In [ ]:
ANALYSIS_DATE_STR = '2025-06-14'

# ── Client config (None = base defaults; 'eastofengland' = client override) ────
CLIENT = None   # or 'eastofengland'

QC_STORE_ID       = None
CORTEX_TEST_N     = 5   # number of signals to test Cortex on

import datetime
analysis_date   = datetime.date.fromisoformat(ANALYSIS_DATE_STR)
THIS_WEEK_END   = str(analysis_date)
THIS_WEEK_START = str(analysis_date - datetime.timedelta(days=6))
LAST_WEEK_END   = str(analysis_date - datetime.timedelta(days=7))
LAST_WEEK_START = str(analysis_date - datetime.timedelta(days=13))
LOOKBACK_START  = str(analysis_date - datetime.timedelta(days=69))
print(f'Window: {LAST_WEEK_START} – {THIS_WEEK_END}')
print(f'Client: {CLIENT or "base defaults"}')

Window: 2025-06-01 – 2025-06-14


In [ ]:
import sys, os, logging, warnings

_root = os.path.abspath(os.path.join('..', '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

# ── Suppress noisy Snowflake connector output ─────────────────────────────────
logging.getLogger('snowflake.connector').setLevel(logging.ERROR)
logging.getLogger('snowflake.snowpark').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='snowflake')

from store_insight.pipeline.config.loader import load_config
from store_insight.pipeline.pipeline.session import create_session
from snowflake.snowpark import functions as F

cfg     = load_config(CLIENT)
print(f'Config loaded: client={cfg["client_name"]}')

session = create_session(cfg)   # prints 'Snowflake session created.'

# ── Resolve table names from config ──────────────────────────────────────────
OSA_TABLE       = cfg['tables']['osa']
HIERARCHY_TABLE = cfg['tables']['hierarchy']
FEEDBACK_TABLE  = cfg['tables']['feedback']

print(f'OSA table       : {OSA_TABLE}')
print(f'Hierarchy table : {HIERARCHY_TABLE}')
print(f'Feedback table  : {FEEDBACK_TABLE}')

osa_raw = (
    session.table(OSA_TABLE)
    .filter(F.col('ANALYSIS_DATE') >= LOOKBACK_START)
    .filter(F.col('ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('SITE_ID').alias('STORE_ID'), F.col('ITEM_ID').alias('ITEM_CODE'),
        F.col('ACTUAL_SALES_VOLUME'), F.col('LOST_SALES_VOLUME'),
        F.col('LOST_SALES_VALUE'), F.col('ANALYSIS_DATE'), F.col('DEPARTMENT_ID'),
    )
)
item = (
    session.table(HIERARCHY_TABLE)
    .select(
        F.col('MASTER_ITEM_ID').alias('ITEM_CODE'), F.col('ITEM_ID'),
        F.col('DEPARTMENT_NAME'), F.col('SUBCLASS_NAME'), F.col('PRODUCT_PROFILE_GROUP_NAME'),
    ).distinct()
)
osa = (
    osa_raw.join(item, on=['ITEM_CODE'], how='left')
    .filter(F.col('DEPARTMENT_NAME').is_not_null())
    .cache_result()
)

feedback = (
    session.table(FEEDBACK_TABLE)
    .filter(F.col('ALERT_ANALYSIS_DATE') >= LAST_WEEK_START)
    .filter(F.col('ALERT_ANALYSIS_DATE') <= THIS_WEEK_END)
    .select(
        F.col('SITE_ID').alias('STORE_ID'),
        F.col('SITE_NAME'),
        F.col('ALERT_TYPE_NAME'),
        F.col('ALERT_ANALYSIS_DATE'),
        F.col('INTERVENTION_ID'),
        F.col('TICK_AND_FLICK_FLAG'),
    )
    .cache_result()
)

qc_data = osa.filter(F.col('STORE_ID') == QC_STORE_ID) if QC_STORE_ID else osa

from store_insight.pipeline.stages._1_detect.detect import detect_signals
from store_insight.pipeline.stages._2_facts.facts import enrich_signals

signals  = detect_signals(
    qc_data,
    last_week_start=LAST_WEEK_START, last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START, this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    feedback=feedback,
    cfg=cfg,
)
enriched = enrich_signals(
    signals, osa,
    last_week_start=LAST_WEEK_START, last_week_end=LAST_WEEK_END,
    this_week_start=THIS_WEEK_START, this_week_end=THIS_WEEK_END,
    lookback_start=LOOKBACK_START,
    feedback=feedback,
    cfg=cfg,
)
print(f'{len(enriched)} enriched signals ready')

ModuleNotFoundError: No module named 'store_insight.pipeline._1_detect'

## Step 2 — Rule-Based Narratives (all signals)

In [ ]:
from store_insight.pipeline.stages._3_investigate.rule_based import generate

narratives = [generate(sig) for sig in enriched]

required = ['issue', 'explanation', 'cost_if_not_fixed', 'recommended_action']
blanks = [(enriched[i]['store_id'], enriched[i]['department'], k)
          for i, n in enumerate(narratives)
          for k in required if not n.get(k)]

print(f'{len(narratives)} narratives generated')
print(f'Blank fields: {len(blanks)}')
if blanks:
    for b in blanks[:5]:
        print(f'  store={b[0]}  dept={b[1]}  missing={b[2]}')

587 narratives generated
Blank fields: 0


## Step 3 — Full Narrative: Top Signal by LSV

In [ ]:
top_idx = max(range(len(enriched)), key=lambda i: enriched[i].get('this_week_lsv') or 0)
top_sig = enriched[top_idx]
top_nar = narratives[top_idx]

print('=' * 60)
print(f"EXAMPLE — store={top_sig['store_id']}  dept={top_sig['department']}")
lsv = top_sig.get('this_week_lsv')
lsv_str = f"£{lsv:,.2f}" if lsv is not None else 'n/a'
print(f"triggers={top_sig['trigger_types']}  lsv={lsv_str}")
print('=' * 60)
print(f"\nISSUE:\n  {top_nar['issue']}")
print(f"\nEXPLANATION:\n{top_nar['explanation']}")
print(f"\nCOST IF NOT FIXED:\n  {top_nar['cost_if_not_fixed']}")
print(f"\nRECOMMENDED ACTION:\n  {top_nar['recommended_action']}")

EXAMPLE — store=214  dept=CIGARETTES
triggers=['VS_CHAIN', 'REPEAT']  lsv=£2,828.42

ISSUE:
  CIGARETTES: OSA is 17.2pp below chain average; 11 items with persistent availability issues

EXPLANATION:
There has been £2,828.42 of lost sales in CIGARETTES this week.
Store OSA this week is 74.0%, vs chain average of 91.2% (17.2pp gap).
11 items in CIGARETTES have triggered repeated availability alerts over the last 10 weeks.
Items with persistent availability issues:
• KINGSIZE STANDARD (DEFAULT): triggered 7 weeks, £2,387.99 cumulative lost sales
• KINGSIZE STANDARD (DEFAULT): triggered 6 weeks, £1,744.06 cumulative lost sales
• KINGSIZE STANDARD (DEFAULT): triggered 7 weeks, £1,575.02 cumulative lost sales
Biggest lost sales contributors this week:
• KINGSIZE STANDARD: £1,499.14 this week (last week £1,398.24, +£100.89)
• SUPERKINGS MILD/LIGHTS: £739.06 this week (last week £484.71, +£254.35)
• SUPERKINGS STANDARD: £459.95 this week (last week £464.32, -£4.38)
Biggest week-on-week increa

## Step 4 — Cortex Narrative (single signal)

In [ ]:
from store_insight.pipeline.stages._3_investigate.cortex import generate as cortex_generate

cortex_nar = cortex_generate(top_sig, session)

if cortex_nar:
    print('CORTEX NARRATIVE')
    print(f"ISSUE:\n  {cortex_nar['issue']}")
    print(f"\nEXPLANATION:\n{cortex_nar['explanation']}")
    print(f"\nCOST IF NOT FIXED:\n  {cortex_nar['cost_if_not_fixed']}")
    print(f"\nRECOMMENDED ACTION:\n  {cortex_nar['recommended_action']}")
else:
    print('Cortex call failed — check output above for error')

CORTEX NARRATIVE
ISSUE:
  CIGARETTES: £2828.42 lost sales this week

EXPLANATION:
There has been £2828.42 of lost sales in CIGARETTES this week. Store OSA this week is 74.0%, vs chain average of 91.2% (17.2pp gap). Worst day for lost sales this week: Friday (£703.17)

COST IF NOT FIXED:
  If this continues at the same rate, the store could lose approximately £11313.68 in the next 4 weeks.

RECOMMENDED ACTION:
  Review availability in CIGARETTES daily. Ensure replenishment orders are raised before 8am and all gaps on high-velocity lines are filled before the store opens. Pay particular attention on Fridays, the worst day for lost sales this week. Address the 11 repeat alert items to prevent further loss of sales.


## Step 5 — Cortex Batch Test (first N signals)

In [ ]:
success, fail = 0, 0

for sig in enriched[:CORTEX_TEST_N]:
    result = cortex_generate(sig, session)
    if result and all(k in result for k in required):
        success += 1
    else:
        fail += 1

print(f'Cortex batch ({CORTEX_TEST_N} signals): {success} success, {fail} fallback')

Cortex batch (5 signals): 5 success, 0 fallback


In [ ]:
enriched[1]

{'store_id': 483,
 'department': 'CIGARETTES',
 'trigger_types': ['WOW'],
 'last_week_osa': 95.29,
 'this_week_osa': 94.28,
 'wow_drift_pp': -1.01,
 'this_week_lsv': 303.65,
 'chain_this_week_osa': None,
 'chain_gap_pp': None,
 'chain_wow_drift_pp': None,
 'repeat_item_count': None,
 'repeat_item_details': [],
 'facts': {'subclass_contributors': [{'subclass_name': 'KINGSIZE STANDARD',
    'this_week_lsv': 199.84,
    'last_week_lsv': 86.05,
    'lsv_change': 113.8},
   {'subclass_name': 'SUPERKINGS MILD/LIGHTS',
    'this_week_lsv': 54.27,
    'last_week_lsv': 61.88,
    'lsv_change': -7.61},
   {'subclass_name': 'KINGSIZE MILD/LIGHTS',
    'this_week_lsv': 35.08,
    'last_week_lsv': 58.75,
    'lsv_change': -23.67}],
  'subclass_changes': [{'subclass_name': 'KINGSIZE STANDARD',
    'this_week_lsv': 199.84,
    'last_week_lsv': 86.05,
    'lsv_change': 113.8},
   {'subclass_name': 'KINGSIZE ULTRA',
    'this_week_lsv': 0.0,
    'last_week_lsv': 0.0,
    'lsv_change': 0.0},
   {'subcla